In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, Dense, Flatten, LSTM, Dropout, Concatenate
from tensorflow.keras.applications import MobileNetV2
import cv2
import librosa

class SpeechDetector:
    def __init__(self, input_shape=(10, 224, 224, 3), audio_length=1000):
        """
        Initialize the speech detection model.
        
        Args:
            input_shape: Shape of video frames input (n_frames, height, width, channels)
            audio_length: Length of audio features
        """
        self.input_shape = input_shape
        self.audio_length = audio_length
        self.model = self._build_model()
        
    def _build_model(self):
        """Build multimodal model architecture combining visual and audio features"""
        # Visual stream - process video frames
        frame_input = Input(shape=self.input_shape)
        
        # Use MobileNetV2 as base model for feature extraction (more efficient than full VGG/ResNet)
        base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
        
        # Process each frame through the base model
        encoded_frames = []
        for i in range(self.input_shape[0]):
            x = tf.keras.layers.Lambda(lambda x: x[:, i])(frame_input)
            x = base_model(x)
            x = Flatten()(x)
            encoded_frames.append(x)
        
        # Combine frame features with LSTM to capture temporal patterns
        visual_features = tf.stack(encoded_frames, axis=1)
        visual_features = LSTM(128, return_sequences=False)(visual_features)
        visual_features = Dense(64, activation='relu')(visual_features)
        
        # Audio stream
        audio_input = Input(shape=(self.audio_length,))
        audio_features = Dense(256, activation='relu')(audio_input)
        audio_features = Dropout(0.3)(audio_features)
        audio_features = Dense(128, activation='relu')(audio_features)
        audio_features = Dense(64, activation='relu')(audio_features)
        
        # Combine visual and audio features
        combined = Concatenate()([visual_features, audio_features])
        combined = Dense(64, activation='relu')(combined)
        combined = Dropout(0.3)(combined)
        combined = Dense(32, activation='relu')(combined)
        output = Dense(1, activation='sigmoid')(combined)
        
        # Create and compile model
        model = Model(inputs=[frame_input, audio_input], outputs=output)
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        
        return model
    
    def extract_features(self, video_path):
        """
        Extract features from a video file.
        
        Args:
            video_path: Path to the video file
            
        Returns:
            video_features: Array of video frames
            audio_features: Array of audio features
        """
        # Open video file
        cap = cv2.VideoCapture(video_path)
        
        # Extract video frames
        frames = []
        count = 0
        while count < self.input_shape[0]:
            ret, frame = cap.read()
            if not ret:
                break
            
            # Resize and preprocess frame
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (self.input_shape[1], self.input_shape[2]))
            frame = frame / 255.0  # Normalize
            frames.append(frame)
            count += 1
            
        cap.release()
        
        # If we didn't get enough frames, pad with zeros
        while len(frames) < self.input_shape[0]:
            frames.append(np.zeros((self.input_shape[1], self.input_shape[2], 3)))
            
        # Extract audio and convert to MFCC features
        y, sr = librosa.load(video_path, sr=None)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
        
        # Ensure audio features are of fixed length
        if mfcc.shape[1] >= self.audio_length:
            mfcc = mfcc[:, :self.audio_length]
        else:
            padding = np.zeros((mfcc.shape[0], self.audio_length - mfcc.shape[1]))
            mfcc = np.concatenate((mfcc, padding), axis=1)
            
        # Flatten MFCCs to 1D array
        mfcc_flat = mfcc.reshape(-1)
        
        # If audio features are too long, truncate
        if len(mfcc_flat) > self.audio_length:
            mfcc_flat = mfcc_flat[:self.audio_length]
        # If too short, pad with zeros
        elif len(mfcc_flat) < self.audio_length:
            mfcc_flat = np.pad(mfcc_flat, (0, self.audio_length - len(mfcc_flat)))
            
        return np.array(frames), mfcc_flat
    
    def train(self, video_paths, labels, validation_split=0.2, epochs=10, batch_size=16):
        """
        Train the model on a set of videos.
        
        Args:
            video_paths: List of paths to video files
            labels: Binary labels (0=not speaking, 1=speaking)
            validation_split: Portion of data to use for validation
            epochs: Number of training epochs
            batch_size: Batch size for training
        """
        # Extract features from all videos
        X_video = []
        X_audio = []
        
        for video_path in video_paths:
            video_features, audio_features = self.extract_features(video_path)
            X_video.append(video_features)
            X_audio.append(audio_features)
            
        X_video = np.array(X_video)
        X_audio = np.array(X_audio)
        y = np.array(labels)
        
        # Train the model
        history = self.model.fit(
            [X_video, X_audio], y,
            validation_split=validation_split,
            epochs=epochs,
            batch_size=batch_size
        )
        
        return history
    
    def predict(self, video_path):
        """
        Predict whether there is speech in a video.
        
        Args:
            video_path: Path to the video file
            
        Returns:
            probability: Probability of speech being present
        """
        video_features, audio_features = self.extract_features(video_path)
        
        # Add batch dimension
        video_features = np.expand_dims(video_features, axis=0)
        audio_features = np.expand_dims(audio_features, axis=0)
        
        # Make prediction
        prediction = self.model.predict([video_features, audio_features])
        return prediction[0][0]
    
    def save(self, model_path):
        """Save the model to disk"""
        self.model.save(model_path)
        
    def load(self, model_path):
        """Load the model from disk"""
        self.model = tf.keras.models.load_model(model_path)


# Example usage
def main():
    # Create model
    detector = SpeechDetector()
    
    # Example training (you would provide actual paths and labels)
    video_paths = ['path/to/video1.mp4', 'path/to/video2.mp4', ...]
    labels = [1, 0, ...]  # 1=speaking, 0=not speaking
    
    # Train model
    history = detector.train(video_paths, labels)
    
    # Save model
    detector.save('speech_detector_model.h5')
    
    # Example prediction
    test_video = 'path/to/test_video.mp4'
    probability = detector.predict(test_video)
    print(f"Probability of speech: {probability:.2f}")
    
    if probability > 0.5:
        print("Speech detected")
    else:
        print("No speech detected")

if __name__ == "__main__":
    main()

ImportError: cannot import name 'MobileNetV1' from 'tensorflow.keras.applications' (/home/tymstr/.local/lib/python3.12/site-packages/keras/_tf_keras/keras/applications/__init__.py)